# Step 5D.2 — F0/F1 Baseline: CIC-IDS2017 та UNSW-NB15 (frozen protocol)

## 1. Frozen protocol constants

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score

ENRICHED_DIR = Path("/srv/ids_research/data/enriched")
RESULTS_DIR = Path.home() / "notebooks" / "2 article" / "release" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TOP_K_FRACTIONS = [0.001, 0.005, 0.01, 0.05]

LGB_PARAMS = dict(
    objective="binary",
    metric="None",              # вимикаємо вбудовану WEIGHTED метрику — early stopping йде за
                                 # власним unweighted AP (feval нижче), інакше критерій відбору
                                 # best_iteration розійшовся б з критерієм фінальної оцінки.
    seed=RANDOM_STATE,
    num_leaves=15,
    learning_rate=0.05,
    min_data_in_leaf=50,
)
NUM_BOOST_ROUND = 500
EARLY_STOPPING_ROUNDS = 30

def unweighted_average_precision(preds, dataset):
    y_true = dataset.get_label()
    return "AP_unweighted", average_precision_score(y_true, preds), True

print(f"Frozen LightGBM params: {LGB_PARAMS}")
print(f"num_boost_round={NUM_BOOST_ROUND}, early_stopping_rounds={EARLY_STOPPING_ROUNDS}")

Frozen LightGBM params: {'objective': 'binary', 'metric': 'None', 'seed': 42, 'num_leaves': 15, 'learning_rate': 0.05, 'min_data_in_leaf': 50}
num_boost_round=500, early_stopping_rounds=30


## 2. Dataset config, loading, binary target construction

In [2]:
DATASET_CONFIG = {
    "CIC": dict(
        file="cic_ids2017_with_cve_context.parquet",
        label_col="attack_label", benign_value="BENIGN",
        time_col="Timestamp",
        boundary_1=pd.Timestamp("2017-07-06 09:28:00"),
        boundary_2=pd.Timestamp("2017-07-07 03:23:00"),
        categorical_cols=["protocol"],
        f0_features=['Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 'Average Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min', 'protocol'],
        f1_additions=['cve_ctx_service', 'cve_ctx_pool_size', 'cve_ctx_n_epss', 'cve_ctx_epss_coverage',
                     'cve_ctx_service_max_epss', 'cve_ctx_n_generic_os_matches', 'cve_ctx_n_specific_product_matches'],
    ),
    "UNSW": dict(
        file="unsw_nb15_with_cve_context_dedup.parquet",   # ВЖЕ дедуплікований (Step 5C.2), не дублюємо тут
        label_col="attack_cat", benign_value="Normal",
        time_col="Stime",
        boundary_1=pd.Timestamp("2015-02-18 05:36:51"),
        boundary_2=pd.Timestamp("2015-02-18 08:58:36"),
        categorical_cols=["state", "proto", "service"],
        f0_features=['state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'Sload', 'Dload',
                    'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth',
                    'res_bdy_len', 'Sjit', 'Djit', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat',
                    'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd',
                    'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm',
                    'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'proto', 'service'],
        f1_additions=['cve_ctx_service', 'cve_ctx_pool_size', 'cve_ctx_epss_coverage',
                     'cve_ctx_n_generic_os_matches', 'cve_ctx_n_specific_product_matches'],
    ),
}

raw = {}
for name, cfg in DATASET_CONFIG.items():
    df = pd.read_parquet(ENRICHED_DIR / cfg["file"])
    df["_is_attack"] = (df[cfg["label_col"]] != cfg["benign_value"]).astype(int)
    if name == "UNSW":
        # ct_ftp_cmd: ' ' -> NaN, потім numeric (Step 5D.1 фінальне рішення). Без імпутації —
        # NaN передається LightGBM напряму.
        df["ct_ftp_cmd"] = pd.to_numeric(df["ct_ftp_cmd"].replace(" ", np.nan), errors="coerce")
    # УВАГА: приведення до category dtype НЕ робиться тут — це відкладено до ПІСЛЯ temporal
    # split (Cell 3), щоб categorical vocabulary формувався ЛИШЕ з train, а не з усього
    # dataset (fit-on-train / transform-on-val-test, той самий принцип, що й scale_pos_weight).
    raw[name] = df
    print(f"{name}: {len(df):,} рядків, Attack={df['_is_attack'].mean():.2%}")

CIC: 2,827,677 рядків, Attack=19.68%
UNSW: 2,059,418 рядків, Attack=4.84%


## 3. Temporal split за frozen межами + ОКРЕМІ assert на кожну межу

In [3]:
splits = {}
categorical_vocab_report = []

for name, cfg in DATASET_CONFIG.items():
    df = raw[name]
    t = pd.to_datetime(df[cfg["time_col"]], unit="s") if cfg["time_col"] == "Stime" else pd.to_datetime(df[cfg["time_col"]])

    train_mask = t < cfg["boundary_1"]
    val_mask = (t >= cfg["boundary_1"]) & (t < cfg["boundary_2"])
    test_mask = t >= cfg["boundary_2"]

    train_df, val_df, test_df = df[train_mask].copy(), df[val_mask].copy(), df[test_mask].copy()
    train_t, val_t, test_t = t[train_mask], t[val_mask], t[test_mask]

    # ДВІ ОКРЕМІ перевірки (не одна об'єднана) — щоб чітко бачити, яка саме межа порушена,
    # якщо колись щось піде не так (напр. дані оновляться, чи файл переплутається).
    assert train_t.max() < val_t.min(), f"{name}: max(train_time)={train_t.max()} НЕ < min(val_time)={val_t.min()}"
    assert val_t.max() < test_t.min(), f"{name}: max(val_time)={val_t.max()} НЕ < min(test_time)={test_t.min()}"

    # ---- Categorical vocabulary: FIT ЛИШЕ на train, TRANSFORM val/test за тим самим набором ----
    # Значення у val/test, яких немає в train, стають NaN (pd.Categorical це робить автоматично) —
    # LightGBM трактує їх як missing, а не як окрему "невідому" категорію. Це запобігає
    # information leakage на рівні препроцесингу (val/test НЕ впливають на те, які категорії
    # взагалі існують для моделі), навіть попри те, що для дерев це не є прямим target leakage.
    categorical_candidates = list(cfg["categorical_cols"])
    if "cve_ctx_service" in df.columns:
        categorical_candidates.append("cve_ctx_service")

    for col in categorical_candidates:
        train_categories = pd.Categorical(train_df[col]).categories
        train_df[col] = pd.Categorical(train_df[col], categories=train_categories)
        val_before_na = val_df[col].isna().sum()
        test_before_na = test_df[col].isna().sum()
        val_df[col] = pd.Categorical(val_df[col], categories=train_categories)
        test_df[col] = pd.Categorical(test_df[col], categories=train_categories)
        val_new_na = int(val_df[col].isna().sum()) - int(val_before_na)
        test_new_na = int(test_df[col].isna().sum()) - int(test_before_na)
        categorical_vocab_report.append(dict(
            dataset=name, column=col, n_train_categories=len(train_categories),
            val_unseen_to_nan=val_new_na, test_unseen_to_nan=test_new_na,
        ))

    splits[name] = dict(train=train_df, val=val_df, test=test_df)
    print(f"{name}: train={len(train_df):,} (attack={train_df['_is_attack'].mean():.2%})  "
          f"val={len(val_df):,} (attack={val_df['_is_attack'].mean():.2%})  "
          f"test={len(test_df):,} (attack={test_df['_is_attack'].mean():.2%})")
    print(f"  ✅ обидві часові межі перевірено окремими assert'ами")

print("\n" + "=" * 70)
print("Categorical vocabulary: fit-on-train / transform-on-val-test — звіт")
print("=" * 70)
display(pd.DataFrame(categorical_vocab_report))

CIC: train=1,979,346 (attack=13.43%)  val=419,191 (attack=37.39%)  test=429,140 (attack=31.22%)
  ✅ обидві часові межі перевірено окремими assert'ами
UNSW: train=1,441,568 (attack=3.42%)  val=308,915 (attack=8.07%)  test=308,935 (attack=8.21%)
  ✅ обидві часові межі перевірено окремими assert'ами

Categorical vocabulary: fit-on-train / transform-on-val-test — звіт


,dataset,column,n_train_categories,val_unseen_to_nan,test_unseen_to_nan
0,CIC,protocol,3,0,0
1,CIC,cve_ctx_service,29,0,0
2,UNSW,state,16,0,0
3,UNSW,proto,135,0,0
4,UNSW,service,13,0,0
5,UNSW,cve_ctx_service,28,0,0


## 4-5. F0/F1 feature contract (categorical/missing-обробка вже застосована в кроці 2)

In [4]:
FEATURE_SETS = {}
for name, cfg in DATASET_CONFIG.items():
    FEATURE_SETS[(name, "F0")] = list(cfg["f0_features"])
    FEATURE_SETS[(name, "F1")] = list(cfg["f0_features"]) + list(cfg["f1_additions"])
    print(f"{name}-F0: {len(FEATURE_SETS[(name,'F0')])} ознак")
    print(f"{name}-F1: {len(FEATURE_SETS[(name,'F1')])} ознак (+{len(cfg['f1_additions'])} CTI)")

# Точна кількість ознак ЗАФІКСОВАНА у Step 5D.1 — будь-яке відхилення означає, що щось
# змінилося (список ознак, дублікат колонки, помилка конфігурації) і потребує розбору.
EXPECTED_FEATURE_COUNTS = {("CIC", "F0"): 77, ("CIC", "F1"): 84, ("UNSW", "F0"): 41, ("UNSW", "F1"): 46}
for key, expected in EXPECTED_FEATURE_COUNTS.items():
    actual = len(FEATURE_SETS[key])
    assert actual == expected, f"{key}: actual={actual}, expected={expected} — feature contract змінився!"
print("\n✅ Точна кількість ознак збігається із зафіксованою Step 5D.1 для всіх 4 моделей")

CIC-F0: 77 ознак
CIC-F1: 84 ознак (+7 CTI)
UNSW-F0: 41 ознак
UNSW-F1: 46 ознак (+5 CTI)

✅ Точна кількість ознак збігається із зафіксованою Step 5D.1 для всіх 4 моделей


## 6. scale_pos_weight — train-only, СПІЛЬНИЙ для F0 і F1 одного датасету

In [5]:
scale_pos_weight_by_dataset = {}
for name in DATASET_CONFIG:
    y_train = splits[name]["train"]["_is_attack"]
    n_pos, n_neg = int(y_train.sum()), int((~y_train.astype(bool)).sum())
    w = n_neg / n_pos
    scale_pos_weight_by_dataset[name] = w
    print(f"{name}: n_pos_train={n_pos:,}, n_neg_train={n_neg:,}, scale_pos_weight={w:.4f}")

CIC: n_pos_train=265,848, n_neg_train=1,713,498, scale_pos_weight=6.4454
UNSW: n_pos_train=49,346, n_neg_train=1,392,222, scale_pos_weight=28.2135


## 7. Єдина функція навчання/оцінювання — використовується для всіх 4 моделей ідентично

In [6]:
def train_and_evaluate(dataset_name, feature_set_name):
    """ІДЕНТИЧНИЙ pipeline для CIC-F0/F1 та UNSW-F0/F1 — відмінність лише в наборі ознак."""
    cfg = DATASET_CONFIG[dataset_name]
    features = FEATURE_SETS[(dataset_name, feature_set_name)]

    train_df, val_df, test_df = splits[dataset_name]["train"], splits[dataset_name]["val"], splits[dataset_name]["test"]
    X_train, y_train = train_df[features], train_df["_is_attack"]
    X_val, y_val = val_df[features], val_df["_is_attack"]
    X_test, y_test = test_df[features], test_df["_is_attack"]

    # ДИНАМІЧНЕ визначення категоріальних колонок за фактичним dtype (не жорстко заданий
    # список) — так і нативні (state/proto/service), і CTI-категоріальні (cve_ctx_service,
    # якщо є у F1) підхоплюються однаково, без окремого правила для кожного типу джерела.
    cat_cols = [c for c in features if X_train[c].dtype.name == "category"]

    w = scale_pos_weight_by_dataset[dataset_name]   # той самий train-only вага для F0 і F1
    params = dict(LGB_PARAMS, scale_pos_weight=w)

    lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols if cat_cols else "auto")
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train, categorical_feature=cat_cols if cat_cols else "auto")

    model = lgb.train(
        params, lgb_train, num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[lgb_val], valid_names=["val"],   # ЛИШЕ val — test НІКОЛИ не передається сюди
        feval=unweighted_average_precision,
        callbacks=[lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
                  lgb.log_evaluation(period=0)],
    )
    best_iter = model.best_iteration   # ЗАФІКСОВАНА кількість дерев, обрана ЛИШЕ на val

    # test-прогноз — ОДИН РАЗ, з уже зафіксованим best_iter, ПІСЛЯ завершення early stopping
    val_scores = model.predict(X_val, num_iteration=best_iter)
    test_scores = model.predict(X_test, num_iteration=best_iter)

    def topk_metrics(y_true, y_score):
        rows = []
        order = np.argsort(-y_score, kind="mergesort")
        y_true_arr = y_true.values
        for frac in TOP_K_FRACTIONS:
            k = max(1, int(round(len(y_true_arr) * frac)))
            top_idx = order[:k]
            tp = y_true_arr[top_idx].sum()
            rows.append(dict(top_pct=f"{frac:.1%}", K=k,
                             precision=tp / k, recall=tp / y_true_arr.sum() if y_true_arr.sum() else np.nan))
        return pd.DataFrame(rows)

    y_pred_test = (test_scores >= 0.5).astype(int)
    result = dict(
        dataset=dataset_name, feature_set=feature_set_name, n_features=len(features),
        best_iteration=best_iter,
        val_ap=average_precision_score(y_val, val_scores),
        val_roc_auc=roc_auc_score(y_val, val_scores),
        test_ap=average_precision_score(y_test, test_scores),
        test_roc_auc=roc_auc_score(y_test, test_scores),
        test_precision=precision_score(y_test, y_pred_test, zero_division=0),
        test_recall=recall_score(y_test, y_pred_test, zero_division=0),
        test_f1=f1_score(y_test, y_pred_test, zero_division=0),
    )
    topk_df = topk_metrics(y_test, test_scores)
    return result, topk_df, model

## 8-11. CIC-F0, CIC-F1, UNSW-F0, UNSW-F1 — та сама функція, різні аргументи

In [7]:
all_results, all_topk, all_models = [], {}, {}
for dataset_name in ["CIC", "UNSW"]:
    for feature_set_name in ["F0", "F1"]:
        print(f"\n{'='*20} {dataset_name}-{feature_set_name} {'='*20}")
        result, topk_df, model = train_and_evaluate(dataset_name, feature_set_name)
        print(f"best_iteration={result['best_iteration']}  test_AP={result['test_ap']:.6f}  "
              f"test_ROC-AUC={result['test_roc_auc']:.4f}")
        all_results.append(result)
        all_topk[(dataset_name, feature_set_name)] = topk_df
        all_models[(dataset_name, feature_set_name)] = model


==================== CIC-F0 ====================
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 265848, number of negative: 1713498
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.261765 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14089
[LightGBM] [Info] Number of data points in the train set: 1979346, number of used features: 69
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.134311 -> initscore=-1.863367
[LightGBM] [Info] Start training from score -1.863367
best_iteration=4  test_AP=0.710369  test_ROC-AUC=0.7406

==================== CIC-F1 ====================
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 26

## 12. Зведена таблиця val/test метрик для всіх 4 моделей

In [8]:
results_df = pd.DataFrame(all_results).set_index(["dataset", "feature_set"])
display(results_df[["n_features", "best_iteration", "val_ap", "val_roc_auc",
                    "test_ap", "test_roc_auc", "test_precision", "test_recall", "test_f1"]])

n_features  best_iteration    val_ap  val_roc_auc  \
dataset feature_set                                                      
CIC     F0                   77               4  0.778550     0.823532   
        F1                   84             118  0.676896     0.765135   
UNSW    F0                   41              33  0.964633     0.997017   
        F1                   46             290  0.979776     0.997882   

                      test_ap  test_roc_auc  test_precision  test_recall  \
dataset feature_set                                                        
CIC     F0           0.710369      0.740553        0.000000     0.000000   
        F1           0.974028      0.982162        0.997403     0.604780   
UNSW    F0           0.973652      0.997671        0.778539     0.999842   
        F1           0.983969      0.998247        0.779184     0.998028   

                      test_f1  
dataset feature_set            
CIC     F0           0.000000  
        F1           0.752984  
UNSW    F0           0.875421  
        F1           0.875132

## 13. Precision/Recall@K (test) для всіх 4 моделей

In [9]:
for (dataset_name, feature_set_name), topk_df in all_topk.items():
    print(f"\n{dataset_name}-{feature_set_name}:")
    display(topk_df)


CIC-F0:


,top_pct,K,precision,recall
0,0.1%,429,1.0,0.003202
1,0.5%,2146,1.0,0.016017
2,1.0%,4291,1.0,0.032026
3,5.0%,21457,1.0,0.160144



CIC-F1:


,top_pct,K,precision,recall
0,0.1%,429,0.997669,0.003194
1,0.5%,2146,0.999534,0.016009
2,1.0%,4291,0.999767,0.032018
3,5.0%,21457,0.999953,0.160136



UNSW-F0:


,top_pct,K,precision,recall
0,0.1%,309,1.000000,0.012187
1,0.5%,1545,1.000000,0.060937
2,1.0%,3089,1.000000,0.121835
3,5.0%,15447,0.991455,0.604047



UNSW-F1:


,top_pct,K,precision,recall
0,0.1%,309,1.000000,0.012187
1,0.5%,1545,1.000000,0.060937
2,1.0%,3089,1.000000,0.121835
3,5.0%,15447,0.999741,0.609095


## 14. ΔAP = F1 − F0 (головна відповідь на питання Step 5)

In [10]:
delta_rows = []
for dataset_name in ["CIC", "UNSW"]:
    ap_f0 = results_df.loc[(dataset_name, "F0"), "test_ap"]
    ap_f1 = results_df.loc[(dataset_name, "F1"), "test_ap"]
    roc_f0 = results_df.loc[(dataset_name, "F0"), "test_roc_auc"]
    roc_f1 = results_df.loc[(dataset_name, "F1"), "test_roc_auc"]
    delta_rows.append(dict(dataset=dataset_name, F0_AP=ap_f0, F1_AP=ap_f1, delta_AP=ap_f1 - ap_f0,
                           F0_ROC_AUC=roc_f0, F1_ROC_AUC=roc_f1, delta_ROC_AUC=roc_f1 - roc_f0))
delta_df = pd.DataFrame(delta_rows).set_index("dataset")
print("=" * 70)
print("ΔAP: чи додає CTI-контекст інформацію понад native flow features?")
print("=" * 70)
display(delta_df)

ΔAP: чи додає CTI-контекст інформацію понад native flow features?


,F0_AP,F1_AP,delta_AP,F0_ROC_AUC,F1_ROC_AUC,delta_ROC_AUC
dataset,,,,,,
CIC,0.710369,0.974028,0.263659,0.740553,0.982162,0.241609
UNSW,0.973652,0.983969,0.010317,0.997671,0.998247,0.000576


## 15. Фінальні leakage/reproducibility перевірки

In [11]:
print("1. F0 і F1 одного датасету використали ОДНАКОВИЙ scale_pos_weight?")
for name in DATASET_CONFIG:
    assert all_models[(name, "F0")].params.get("scale_pos_weight") == \
           all_models[(name, "F1")].params.get("scale_pos_weight"), f"{name}: F0/F1 мають різну вагу!"
    print(f"   ✅ {name}: {scale_pos_weight_by_dataset[name]:.4f} для F0 і F1")

print("\n2. cve_ctx_* НЕ потрапили у F0 жодного датасету?")
for name in DATASET_CONFIG:
    f0_cols = set(FEATURE_SETS[(name, "F0")])
    assert not any(c.startswith("cve_ctx_") for c in f0_cols), f"{name}: cve_ctx_* просочився у F0!"
    print(f"   ✅ {name}-F0: жодного cve_ctx_*")

print("\n3. F1 = F0 + ЛИШЕ frozen CTI-додавання (без випадкових зайвих ознак)?")
for name, cfg in DATASET_CONFIG.items():
    expected_f1 = set(cfg["f0_features"]) | set(cfg["f1_additions"])
    actual_f1 = set(FEATURE_SETS[(name, "F1")])
    assert expected_f1 == actual_f1, f"{name}: F1 не відповідає F0+additions!"
    print(f"   ✅ {name}-F1 == F0 + {len(cfg['f1_additions'])} CTI ознак, точно")

print("\n4. best_iteration обрано на validation (перевірка: val_ap узгоджений з best_iteration)?")
for (name, fs), model in all_models.items():
    assert model.best_iteration is not None
    print(f"   ✅ {name}-{fs}: best_iteration={model.best_iteration}")

print("\n5. Categorical vocabulary: val/test категорії — ПІДМНОЖИНА train категорій (fit-on-train)?")
for name, cfg in DATASET_CONFIG.items():
    categorical_candidates = list(cfg["categorical_cols"])
    if "cve_ctx_service" in splits[name]["train"].columns:
        categorical_candidates.append("cve_ctx_service")
    for col in categorical_candidates:
        train_cats = set(splits[name]["train"][col].cat.categories)
        val_cats = set(splits[name]["val"][col].dropna().unique())
        test_cats = set(splits[name]["test"][col].dropna().unique())
        assert val_cats <= train_cats, f"{name}.{col}: val містить категорії поза train vocabulary — {val_cats - train_cats}"
        assert test_cats <= train_cats, f"{name}.{col}: test містить категорії поза train vocabulary — {test_cats - train_cats}"
        print(f"   ✅ {name}.{col}: val/test категорії ⊆ train vocabulary ({len(train_cats)} категорій)")

print("\n✅ Усі перевірки пройдено.")

1. F0 і F1 одного датасету використали ОДНАКОВИЙ scale_pos_weight?
   ✅ CIC: 6.4454 для F0 і F1
   ✅ UNSW: 28.2135 для F0 і F1

2. cve_ctx_* НЕ потрапили у F0 жодного датасету?
   ✅ CIC-F0: жодного cve_ctx_*
   ✅ UNSW-F0: жодного cve_ctx_*

3. F1 = F0 + ЛИШЕ frozen CTI-додавання (без випадкових зайвих ознак)?
   ✅ CIC-F1 == F0 + 7 CTI ознак, точно
   ✅ UNSW-F1 == F0 + 5 CTI ознак, точно

4. best_iteration обрано на validation (перевірка: val_ap узгоджений з best_iteration)?
   ✅ CIC-F0: best_iteration=4
   ✅ CIC-F1: best_iteration=118
   ✅ UNSW-F0: best_iteration=33
   ✅ UNSW-F1: best_iteration=290

5. Categorical vocabulary: val/test категорії — ПІДМНОЖИНА train категорій (fit-on-train)?
   ✅ CIC.protocol: val/test категорії ⊆ train vocabulary (3 категорій)
   ✅ CIC.cve_ctx_service: val/test категорії ⊆ train vocabulary (29 категорій)
   ✅ UNSW.state: val/test категорії ⊆ train vocabulary (16 категорій)
   ✅ UNSW.proto: val/test категорії ⊆ train vocabulary (135 категорій)
   ✅ UNSW.s

## 16. Збереження заморожених результатів

In [12]:
results_df.reset_index().to_csv(RESULTS_DIR / "step5d2_f0_f1_results_h_flow.csv", index=False)
delta_df.reset_index().to_csv(RESULTS_DIR / "step5d2_delta_ap.csv", index=False)
for (name, fs), topk_df in all_topk.items():
    topk_df.to_csv(RESULTS_DIR / f"step5d2_topk_{name}_{fs}.csv", index=False)

print(f"Збережено в {RESULTS_DIR}:")
for f in sorted(RESULTS_DIR.glob("step5d2_*")):
    print(f"  {f.name}")

Збережено в /home/ivan_r_opirskyy/notebooks/2 article/release/results:
  step5d2_delta_ap.csv
  step5d2_f0_f1_results_h_flow.csv
  step5d2_topk_CIC_F0.csv
  step5d2_topk_CIC_F1.csv
  step5d2_topk_UNSW_F0.csv
  step5d2_topk_UNSW_F1.csv
